# Phase 3 - Exploring the UNSW-NB15 Dataset (EDA)

**EDA = Exploratory Data Analysis.** Before building any model, we look at the data to understand:

1. How big is it? (rows and columns)
2. What do the columns mean?
3. Are there missing values we must fix?
4. How balanced are the classes? (normal vs attack) - this affects how we judge the model.
5. What are the attack types?

Run each cell below top to bottom (click a cell, press `Shift+Enter`).

> **Note on the swapped files:** in our copy of the dataset the filenames are swapped, so the
> file named `UNSW_NB15_testing-set.csv` actually holds the larger *training* split (175,341 rows)
> and `UNSW_NB15_training-set.csv` holds the smaller *testing* split (82,332 rows). We load them by
> their real role below, matching the standard benchmark (larger split = training).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Folder where we save charts for the report
REPORT_DIR = os.path.join("..", "report")
os.makedirs(REPORT_DIR, exist_ok=True)

print("Libraries loaded.")

## Step 1 - Load the data

We load the two CSVs by their real role (remember the filenames are swapped).
`train_df` is what we'll learn from; `test_df` is held back to judge the model fairly.

In [ ]:
DATA_DIR = os.path.join("..", "data")

# Filenames are swapped in this dataset copy, so we assign by real role:
train_df = pd.read_csv(os.path.join(DATA_DIR, "UNSW_NB15_testing-set.csv"))   # 175,341 rows
test_df  = pd.read_csv(os.path.join(DATA_DIR, "UNSW_NB15_training-set.csv"))   # 82,332 rows

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

## Step 2 - Column types and missing values

We check the data types (numeric vs text) and whether anything is missing.
The 3 text columns (`proto`, `service`, `state`) will need encoding into numbers later (Phase 4).

In [ ]:
# Data types
print("Column data types:")
print(train_df.dtypes.value_counts())
print()

# Which columns are text (categorical)?
categorical_cols = train_df.select_dtypes(include="object").columns.tolist()
print("Text (categorical) columns:", categorical_cols)
print()

# Missing values (should be 0 for this dataset)
missing = train_df.isnull().sum()
print("Total missing values in train:", int(missing.sum()))
print("Total missing values in test :", int(test_df.isnull().sum().sum()))

## Step 3 - Class balance (normal vs attack)

`label` is our main target: `0` = normal, `1` = attack. If one class hugely outnumbers the other,
plain accuracy can be misleading, so we always check this first.

In [ ]:
counts = train_df["label"].value_counts().sort_index()
pct = train_df["label"].value_counts(normalize=True).sort_index() * 100
print("Label counts (train):")
for k in counts.index:
    name = "Normal (0)" if k == 0 else "Attack (1)"
    print(f"  {name}: {counts[k]:,}  ({pct[k]:.1f}%)")

plt.figure(figsize=(6, 4))
ax = sns.countplot(x="label", data=train_df, hue="label", palette="Set2", legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Normal (0)", "Attack (1)"])
ax.set_title("Class balance: Normal vs Attack (train)")
ax.set_xlabel("")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "class_balance.png"), dpi=120)
plt.show()

## Step 4 - Attack categories

`attack_cat` tells us *which kind* of attack each record is. This is used later for the
multiclass version (Phase 8). Some attack types are rare - good to know in advance.

In [ ]:
cat_counts = train_df["attack_cat"].value_counts()
print("Attack category counts (train):")
print(cat_counts)

plt.figure(figsize=(9, 4))
ax = sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index,
                 palette="viridis", legend=False)
ax.set_title("Records per attack category (train)")
ax.set_xlabel("Number of records")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "attack_categories.png"), dpi=120)
plt.show()

## Step 5 - A quick look at feature scales

Neural networks train badly when features are on wildly different scales (e.g. `dur` in seconds
vs `sbytes` in millions). The summary below shows that spread - it's *why* we scale everything to
0-1 in Phase 4.

In [ ]:
# Show min/max/mean for a few example numeric features to see the scale differences
example_features = ["dur", "spkts", "sbytes", "dbytes", "rate", "sload"]
train_df[example_features].describe().T[["min", "mean", "max"]]

## Takeaways (what this tells us for the next phases)

- **Data is clean:** no missing values, so no imputation needed.
- **3 categorical columns** (`proto`, `service`, `state`) must be encoded to numbers -> Phase 4.
- **Feature scales vary massively** -> we must scale to 0-1 before feeding the network -> Phase 4.
- **Class balance** tells us to report precision/recall/F1, not just accuracy -> Phase 7.
- Two charts were saved to the `report/` folder for the final write-up.

Next: **Phase 4 - build the preprocessing pipeline** (`src/preprocess.py`).